# V7C – CatBoost per Position

**Why CatBoost?**  
- Handles categoricals natively via ordered target statistics — learns team strength in a way that generalises across seasons  
- Handles NaN natively — no imputation needed for xG columns  
- No OrdinalEncoder needed — pass `team`/`opponent_team` as raw strings

**Train:** 2016-17 → 2023-24 &nbsp;|&nbsp; **Val:** 2023-24 &nbsp;|&nbsp; **Test:** 2024-25  
**Models saved to:** `models_v7c/`


In [1]:
import os, warnings, multiprocessing
import numpy as np
import pandas as pd
import joblib
import optuna
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

print('pandas:', pd.__version__)
print('catboost:', __import__('catboost').__version__)
print('optuna:', optuna.__version__)


pandas: 2.3.2
catboost: 1.2.10
optuna: 4.8.0


In [2]:
# ── Device selection ─────────────────────────────────────────────────────────
try:
    import torch
    if torch.backends.mps.is_available():
        device = "mps"
    elif torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
except ImportError:
    device = "cpu"

n_cores = multiprocessing.cpu_count()
OPTUNA_N_JOBS = 2 if device == "cuda" else 1
CB_TASK_TYPE  = "GPU" if device == "cuda" else "CPU"

print(f"device        : {device}")
print(f"cpu cores     : {n_cores}")
print(f"CatBoost task : {CB_TASK_TYPE}")
print(f"Optuna n_jobs : {OPTUNA_N_JOBS}")
if device in ("mps",):
    print("Note: CatBoost does not support Apple Metal — running on CPU.")


device        : mps
cpu cores     : 12
CatBoost task : CPU
Optuna n_jobs : 1
Note: CatBoost does not support Apple Metal — running on CPU.


In [3]:
DATA_DIR   = os.path.abspath('.')
MODELS_DIR = os.path.join(DATA_DIR, 'models_v7c')
os.makedirs(MODELS_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(DATA_DIR, 'train_v7.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test_v7.csv')

LABEL     = 'future_points'
CAT_COLS  = ['team', 'opponent_team']

N_TRIALS   = 100
N_CV_FOLDS = 5

CLIP_RANGE = {
    'GK':  (-4, 20),
    'DEF': (-4, 25),
    'MID': (-4, 25),
    'FWD': (-4, 25),
}

print('Config ready. Models dir:', MODELS_DIR)


Config ready. Models dir: /Users/yuvalgarzon/FinalProject_AutomaticChempion/Automatic-Champion/weekly_model_data/V7/models_v7c


## 1 · Load Data

In [4]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

# CatBoost needs cat columns as strings
for col in CAT_COLS:
    train[col] = train[col].astype(str)
    test[col]  = test[col].astype(str)

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Train seasons: {sorted(train["season"].unique())}')
print()
print('Rows per position:')
print(pd.DataFrame({
    'Train': train.groupby('position').size(),
    'Test':  test.groupby('position').size(),
}))


Train: (190825, 125)  |  Test: (26499, 125)
Train seasons: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']

Rows per position:
          Train   Test
position              
DEF       64503   8878
FWD       25698   2931
GK        20845   2787
MID       79779  11903


## 2 · Feature Definitions

In [5]:
BASE_FEATURES = [
    'was_home', 'team', 'opponent_team',
    'minutes', 'starts',
    'value', 'selected', 'transfers_balance',
    'ict_index', 'influence', 'creativity', 'threat',
    'bps', 'bonus',
    'xP',
    'expected_goals', 'expected_assists',
    'expected_goal_involvements', 'expected_goals_conceded',
    'team_goals_scored', 'team_goals_conceded',
]

ROLL_BASE = [
    'total_points_last',  'total_points_mean3',  'total_points_mean5',  'total_points_mean7',
    'minutes_last',       'minutes_mean3',        'minutes_mean5',       'minutes_mean7',
    'starts_last',        'starts_mean3',         'starts_mean5',
    'ict_index_last',     'ict_index_mean3',      'ict_index_mean5',     'ict_index_mean7',
    'influence_last',     'influence_mean3',      'influence_mean5',
    'creativity_last',    'creativity_mean3',     'creativity_mean5',
    'threat_last',        'threat_mean3',         'threat_mean5',
    'bps_last',           'bps_mean3',            'bps_mean5',
    'bonus_last',         'bonus_mean3',          'bonus_mean5',
    'value_last',         'value_mean3',
    'selected_last',      'selected_mean3',
    'transfers_balance_last', 'transfers_balance_mean3',
    'xP_last',            'xP_mean3',             'xP_mean5',
    'expected_goals_last',             'expected_goals_mean3',             'expected_goals_mean5',
    'expected_assists_last',           'expected_assists_mean3',           'expected_assists_mean5',
    'expected_goal_involvements_last', 'expected_goal_involvements_mean3', 'expected_goal_involvements_mean5',
    'expected_goals_conceded_last',    'expected_goals_conceded_mean3',    'expected_goals_conceded_mean5',
    'team_goals_scored_last',   'team_goals_scored_mean3',   'team_goals_scored_mean5',
    'team_goals_conceded_last', 'team_goals_conceded_mean3', 'team_goals_conceded_mean5',
]

POS_EXTRAS = {
    'GK': [
        'saves', 'saves_last', 'saves_mean3', 'saves_mean5', 'saves_mean7',
        'clean_sheets', 'clean_sheets_last', 'clean_sheets_mean3', 'clean_sheets_mean5',
        'goals_conceded', 'goals_conceded_last', 'goals_conceded_mean3', 'goals_conceded_mean5',
    ],
    'DEF': [
        'clean_sheets', 'clean_sheets_last', 'clean_sheets_mean3', 'clean_sheets_mean5',
        'goals_conceded', 'goals_conceded_last', 'goals_conceded_mean3', 'goals_conceded_mean5',
        'goals_scored', 'goals_scored_last', 'goals_scored_mean3',
        'assists', 'assists_last', 'assists_mean3',
    ],
    'MID': [
        'goals_scored', 'goals_scored_last', 'goals_scored_mean3', 'goals_scored_mean5',
        'assists', 'assists_last', 'assists_mean3', 'assists_mean5',
    ],
    'FWD': [
        'goals_scored', 'goals_scored_last', 'goals_scored_mean3', 'goals_scored_mean5',
        'assists', 'assists_last', 'assists_mean3', 'assists_mean5',
    ],
}

def get_features(pos, tr, te):
    all_feats = BASE_FEATURES + ROLL_BASE + POS_EXTRAS.get(pos, [])
    available = [f for f in all_feats if f in tr.columns and f in te.columns]
    seen = set()
    return [f for f in available if not (f in seen or seen.add(f))]

for pos in ['GK', 'DEF', 'MID', 'FWD']:
    feats = get_features(pos, train, test)
    cats  = [c for c in CAT_COLS if c in feats]
    print(f'{pos}: {len(feats)} features  ({len(cats)} categorical)')


GK: 88 features  (2 categorical)
DEF: 89 features  (2 categorical)
MID: 86 features  (2 categorical)
FWD: 86 features  (2 categorical)


## 3 · Time-Based Validation Split

In [6]:
last_season     = train['season'].sort_values().unique()[-1]
inner_train     = train[train['season'] != last_season].copy()
val             = train[train['season'] == last_season].copy()

print(f'Inner train: {sorted(inner_train["season"].unique())}  ({len(inner_train):,} rows)')
print(f'Validation:  {sorted(val["season"].unique())}           ({len(val):,} rows)')
print(f'Test:        {sorted(test["season"].unique())}           ({len(test):,} rows)')


Inner train: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']  (161,965 rows)
Validation:  ['2023-24']           (28,860 rows)
Test:        ['2024-25']           (26,499 rows)


## 4 · Optuna-Tuned CatBoost per Position

In [7]:
summary_rows = []

for pos in ['GK', 'DEF', 'MID', 'FWD']:
    print(f'\n{"="*65}')
    print(f'  {pos}  —  {N_TRIALS} Optuna trials  |  {N_CV_FOLDS}-fold TSCV')
    print(f'{"="*65}')

    feats    = get_features(pos, train, test)
    cat_cols = [c for c in CAT_COLS if c in feats]

    it_pos  = inner_train[inner_train['position'] == pos].copy()
    val_pos = val[val['position'] == pos].copy()
    te_pos  = test[test['position'] == pos].copy()

    X_it,  y_it  = it_pos[feats],  it_pos[LABEL].astype(float)
    X_val, y_val = val_pos[feats], val_pos[LABEL].astype(float)
    X_te,  y_te  = te_pos[feats],  te_pos[LABEL].astype(float)

    lo, hi = CLIP_RANGE[pos]

    def objective(trial):
        params = dict(
            iterations        = trial.suggest_int('iterations', 200, 800),
            learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            depth             = trial.suggest_int('depth', 4, 10),
            l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            bagging_temperature = trial.suggest_float('bagging_temperature', 0.0, 1.0),
            random_strength   = trial.suggest_float('random_strength', 0.0, 1.0),
            task_type         = CB_TASK_TYPE,
            random_seed       = 42,
            verbose           = False,
        )
        tscv = TimeSeriesSplit(n_splits=N_CV_FOLDS)
        maes = []
        for tr_idx, vl_idx in tscv.split(X_it):
            pool_tr = Pool(X_it.iloc[tr_idx], y_it.iloc[tr_idx], cat_features=cat_cols)
            pool_vl = Pool(X_it.iloc[vl_idx], cat_features=cat_cols)
            m = CatBoostRegressor(**params)
            m.fit(pool_tr)
            pred = m.predict(pool_vl).clip(lo, hi)
            maes.append(mean_absolute_error(y_it.iloc[vl_idx], pred))
        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=OPTUNA_N_JOBS, show_progress_bar=True)

    best = study.best_params
    best.update({'task_type': CB_TASK_TYPE, 'random_seed': 42, 'verbose': False})

    pool_it  = Pool(X_it,  y_it,  cat_features=cat_cols)
    pool_val = Pool(X_val, cat_features=cat_cols)
    pool_te  = Pool(X_te,  cat_features=cat_cols)

    model = CatBoostRegressor(**best)
    model.fit(pool_it)

    val_pred = model.predict(pool_val).clip(lo, hi)
    te_pred  = model.predict(pool_te).clip(lo, hi)

    val_mae  = mean_absolute_error(y_val, val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    te_mae   = mean_absolute_error(y_te,  te_pred)
    te_rmse  = np.sqrt(mean_squared_error(y_te,  te_pred))

    print(f'  Val  MAE={val_mae:.4f}  RMSE={val_rmse:.4f}')
    print(f'  Test MAE={te_mae:.4f}  RMSE={te_rmse:.4f}')
    print(f'  Best params: { {k:v for k,v in best.items() if k not in ["task_type","verbose"]} }')

    model_path = os.path.join(MODELS_DIR, f'model_{pos}.cbm')
    model.save_model(model_path)
    print(f'  Saved -> {model_path}')

    summary_rows.append({
        'position':  pos,
        'n_features': len(feats),
        'n_train':   len(X_it),
        'val_MAE':   round(val_mae, 4),
        'val_RMSE':  round(val_rmse, 4),
        'test_MAE':  round(te_mae,  4),
        'test_RMSE': round(te_rmse, 4),
        '_model': model, '_pos': pos,
        '_feats': feats, '_cat_cols': cat_cols,
        '_Xte': X_te, '_yte': y_te,
    })

print('\nAll models trained and saved.')



  GK  —  100 Optuna trials  |  5-fold TSCV


  0%|          | 0/100 [00:00<?, ?it/s]

  Val  MAE=0.6111  RMSE=1.3953
  Test MAE=0.7295  RMSE=1.6028
  Best params: {'iterations': 216, 'learning_rate': 0.03389287257770007, 'depth': 10, 'l2_leaf_reg': 8.838977469886943, 'bagging_temperature': 0.3334827205592911, 'random_strength': 0.740648882196332, 'random_seed': 42}
  Saved -> /Users/yuvalgarzon/FinalProject_AutomaticChempion/Automatic-Champion/weekly_model_data/V7/models_v7c/model_GK.cbm

  DEF  —  100 Optuna trials  |  5-fold TSCV


  0%|          | 0/100 [00:00<?, ?it/s]

  Val  MAE=1.0468  RMSE=1.9435
  Test MAE=1.0278  RMSE=1.8470
  Best params: {'iterations': 589, 'learning_rate': 0.013268770996682317, 'depth': 10, 'l2_leaf_reg': 9.202702320626361, 'bagging_temperature': 0.23932860221265495, 'random_strength': 0.8733233402360137, 'random_seed': 42}
  Saved -> /Users/yuvalgarzon/FinalProject_AutomaticChempion/Automatic-Champion/weekly_model_data/V7/models_v7c/model_DEF.cbm

  MID  —  100 Optuna trials  |  5-fold TSCV


  0%|          | 0/100 [00:00<?, ?it/s]

  Val  MAE=0.9401  RMSE=1.9430
  Test MAE=0.9998  RMSE=1.9622
  Best params: {'iterations': 618, 'learning_rate': 0.012138488081383566, 'depth': 10, 'l2_leaf_reg': 6.261867720063693, 'bagging_temperature': 0.9037687933082572, 'random_strength': 0.9965646479342263, 'random_seed': 42}
  Saved -> /Users/yuvalgarzon/FinalProject_AutomaticChempion/Automatic-Champion/weekly_model_data/V7/models_v7c/model_MID.cbm

  FWD  —  100 Optuna trials  |  5-fold TSCV


  0%|          | 0/100 [00:00<?, ?it/s]

  Val  MAE=1.0077  RMSE=2.0990
  Test MAE=1.1534  RMSE=2.2218
  Best params: {'iterations': 531, 'learning_rate': 0.010002671167067899, 'depth': 10, 'l2_leaf_reg': 9.704837718860906, 'bagging_temperature': 0.411693433344049, 'random_strength': 0.8519906330119248, 'random_seed': 42}
  Saved -> /Users/yuvalgarzon/FinalProject_AutomaticChempion/Automatic-Champion/weekly_model_data/V7/models_v7c/model_FWD.cbm

All models trained and saved.


## 5 · Feature Importance

In [8]:
fi_rows = []
for row in summary_rows:
    model    = row['_model']
    pos      = row['_pos']
    feats    = row['_feats']
    cat_cols = row['_cat_cols']
    X_te     = row['_Xte']
    y_te     = row['_yte']

    imp_vals = model.get_feature_importance(Pool(X_te, cat_features=cat_cols))
    imp = pd.DataFrame({
        'feature':    feats,
        'importance': imp_vals,
    }).sort_values('importance', ascending=False)

    print(f'\n── {pos} – Top 15 features ──────────────────────────────────')
    print(imp.head(15).to_string(index=False))

    for _, r in imp.iterrows():
        fi_rows.append({'position': pos, 'feature': r['feature'], 'importance': r['importance']})

fi_df = pd.DataFrame(fi_rows)
fi_df.to_csv(os.path.join(DATA_DIR, 'feature_importance_v7c.csv'), index=False)
print('\nSaved: feature_importance_v7c.csv')



── GK – Top 15 features ──────────────────────────────────
                feature  importance
                minutes   10.099031
                    bps    8.160096
              ict_index    4.731756
              influence    3.535878
                     xP    2.556009
            value_mean3    2.464525
transfers_balance_mean3    2.324139
 transfers_balance_last    2.320063
        ict_index_mean7    2.314577
                   team    2.085060
team_goals_scored_mean3    2.077785
team_goals_scored_mean5    2.063291
 expected_assists_mean5    1.886431
               selected    1.826691
                  value    1.771058

── DEF – Top 15 features ──────────────────────────────────
                feature  importance
                minutes    7.371543
                    bps    4.593104
              ict_index    4.565891
               was_home    3.518607
                   team    3.518432
                  value    3.153713
                     xP    2.919502
              i

## 6 · Final Summary vs V3 & V7

In [9]:
v3 = {'GK': 0.7717, 'DEF': 1.0237, 'MID': 1.0052, 'FWD': 1.1659}
v7 = {'GK': 0.7547, 'DEF': 1.0328, 'MID': 1.0082, 'FWD': 1.1670}

print('=' * 75)
print('COMPARISON  —  Test MAE on 2024-25 season')
print('=' * 75)
print(f'{"Pos":4}  {"V3 MAE":>8} {"V7 MAE":>8} {"V7C MAE":>8}  {"V3→V7C Δ":>10}  BEST')
print('-' * 75)

for row in summary_rows:
    pos     = row['position']
    v7c_mae = row['test_MAE']
    delta   = v7c_mae - v3[pos]
    best    = min(v3[pos], v7[pos], v7c_mae)
    winner  = 'V3' if best == v3[pos] else ('V7' if best == v7[pos] else 'V7C')
    print(f'{pos:4}  {v3[pos]:>8.4f} {v7[pos]:>8.4f} {v7c_mae:>8.4f}  {delta:>+10.4f}  {winner}')

print('=' * 75)
print()
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith('_')}
    for r in summary_rows
])
display(summary_df)


COMPARISON  —  Test MAE on 2024-25 season
Pos     V3 MAE   V7 MAE  V7C MAE    V3→V7C Δ  BEST
---------------------------------------------------------------------------
GK      0.7717   0.7547   0.7295     -0.0422  V7C
DEF     1.0237   1.0328   1.0278     +0.0041  V3
MID     1.0052   1.0082   0.9998     -0.0054  V7C
FWD     1.1659   1.1670   1.1534     -0.0125  V7C



,position,n_features,n_train,val_MAE,val_RMSE,test_MAE,test_RMSE
0,GK,88,17532,0.6111,1.3953,0.7295,1.6028
1,DEF,89,55170,1.0468,1.9435,1.0278,1.8470
2,MID,86,67306,0.9401,1.9430,0.9998,1.9622
3,FWD,86,21957,1.0077,2.0990,1.1534,2.2218
